# Migrando pra PyTorch

Continuação de `01_tensores_fundamentos_censo.ipynb`. Ali aprendemos tensor (escalar → vetor → matriz → rank-3) e treinamos um neurônio escrevendo o gradiente à mão, com NumPy. Aqui refazemos os mesmos cálculos com **PyTorch**, usando os mesmos dados reais (Fortaleza e os 20 setores censitários mais populosos, `censo_setor_idade_sexo`) — o objetivo é ver exatamente o que muda (e o que não muda) ao trocar de biblioteca.

**O que muda:** `torch.tensor` no lugar de `np.array`; os gradientes deixam de ser escritos à mão — `loss.backward()` calcula tudo sozinho (*autograd*); e o treino idiomático usa `nn.Module` + `torch.optim`, não um loop escrito do zero.

**O que não muda:** a matemática. `X @ w + b`, sigmoide, erro quadrático médio, gradiente descendente — é a mesma coisa; PyTorch só automatiza a parte de calcular derivadas.

In [ ]:
import torch
import numpy as np
torch.manual_seed(0)
print("torch:", torch.__version__)

## 1. `torch.tensor` no lugar de `np.array`

Mesmos dados do Passo 5 do notebook 01 (5 regiões, % população 35-45 e razão de sedentarismo). A sintaxe é quase idêntica — a diferença aparece no `dtype` (PyTorch é mais rígido: por padrão usa `float32`, NumPy usa `float64`) e no fato de o tensor poder viver na GPU (`device`), o que não existe em NumPy.

In [ ]:
regioes = ["Sudeste", "Nordeste", "Sul", "Centro-Oeste", "Norte"]

pct_pop_35_45 = torch.tensor([0.18, 0.16, 0.17, 0.16, 0.14])
taxa_sedentarismo_regional = torch.tensor([0.335, 0.40, 0.30, 0.365, 0.425])
taxa_sedentarismo_nacional = torch.tensor(0.34)

print(pct_pop_35_45, "| dtype:", pct_pop_35_45.dtype, "| device:", pct_pop_35_45.device)

razao_sedentarismo = taxa_sedentarismo_regional / taxa_sedentarismo_nacional
entradas = torch.stack([pct_pop_35_45, razao_sedentarismo], dim=1)  # dim= no lugar de axis=
print("entradas shape:", entradas.shape)

## 2. O mesmo neurônio do Passo 5-6, sem treino

Reaproveitando os pesos escolhidos à mão no notebook 01 (`[0.4, 0.6]`, viés `-0.3`) — os números devem bater exatamente com o que já tínhamos calculado em NumPy.

In [ ]:
pesos = torch.tensor([0.4, 0.6])
vies = torch.tensor(-0.3)

score_linear = entradas @ pesos + vies
prioridade = torch.sigmoid(score_linear)

for regiao, score, p in zip(regioes, score_linear, prioridade):
    print(f"{regiao:<13} score={score.item():.3f}   prioridade={p.item():.3f}")

# conferência: os mesmos valores do notebook 01 (Sudeste=0.590, Nordeste=0.615, Sul=0.574, Centro-Oeste=0.601, Norte=0.624)

## 3. Autograd: o gradiente que não escrevemos mais à mão

No notebook 01, Passo 8, escrevemos manualmente `dL/da`, `da/dz`, `dz/dw` (regra da cadeia). Em PyTorch, basta marcar `requires_grad=True` nos tensores que queremos ajustar, chamar `.backward()` no valor da perda, e o autograd percorre o grafo de operações sozinho.

Prova de que bate com a conta manual: calculamos o gradiente para as mesmas 5 regiões, comparando com a fórmula do Passo 8 do notebook 01.

In [ ]:
w = torch.tensor([0.4, 0.6], requires_grad=True)
b = torch.tensor(-0.3, requires_grad=True)
y_fake = torch.zeros(5)  # alvo arbitrário, só para ilustrar o cálculo do gradiente

z = entradas @ w + b
a = torch.sigmoid(z)
perda = torch.mean((a - y_fake) ** 2)

perda.backward()  # autograd calcula tudo
grad_w_autograd = w.grad.clone()

# a mesma conta feita à mão no notebook 01 (Passo 8), com numpy, para comparar
entradas_np = entradas.detach().numpy()
a_np = a.detach().numpy()
n = len(y_fake)
dL_da = 2 * (a_np - 0) / n
da_dz = a_np * (1 - a_np)
grad_w_manual = entradas_np.T @ (dL_da * da_dz)

print("gradiente via autograd :", grad_w_autograd.numpy())
print("gradiente calculado à mão (numpy):", grad_w_manual)

## 4. Dado real: a matriz de Fortaleza (Passo 7) e o tensor rank-3 dos setores (Passo 9)

Mesmos números reais do notebook 01 — `censo_setor_idade_sexo` (Supabase, projeto `epgedaiukjippepujuzc`), Fortaleza (`2304400`), consulta em 2026-08-26. A única mudança é o construtor (`torch.tensor` em vez de `np.array`) e o nome do parâmetro de eixo (`dim=` em vez de `axis=`).

In [ ]:
fortaleza_pop_total = 2_424_722
fortaleza = torch.tensor([
    [175_895, 176_933],   # 15-24
    [282_788, 316_494],   # 25-39 (core fitness)
    [293_456, 363_678],   # 40-59
    [140_797, 220_858],   # 60+
], dtype=torch.float32)

print("Fortaleza shape:", fortaleza.shape, "| soma:", int(fortaleza.sum()))

# tensor rank-3: 20 setores censitários mais populosos de Fortaleza x 4 faixas x 2 sexos
setores_id = [
    "230440005180085", "230440005190223", "230440005160271", "230440005200136",
    "230440005190173", "230440005200161", "230440005200130", "230440005180352",
    "230440005190162", "230440005140152", "230440005200047", "230440005220283",
    "230440005190036", "230440005170200", "230440005190260", "230440005180388",
    "230440005190137", "230440005180237", "230440005230265", "230440005200104",
]
setores = torch.tensor([
    [[170, 211], [851, 993], [373, 459], [62, 114]],
    [[279, 336], [433, 484], [337, 431], [115, 154]],
    [[141, 172], [448, 504], [361, 395], [150, 225]],
    [[193, 225], [355, 384], [261, 280], [63, 65]],
    [[131, 140], [276, 375], [279, 317], [112, 143]],
    [[124, 131], [326, 349], [272, 327], [106, 166]],
    [[105, 135], [277, 327], [263, 269], [78, 88]],
    [[79, 80], [347, 419], [210, 218], [82, 139]],
    [[86, 114], [212, 290], [258, 302], [108, 135]],
    [[109, 113], [229, 257], [250, 302], [90, 157]],
    [[105, 134], [311, 342], [222, 260], [57, 95]],
    [[127, 125], [241, 256], [216, 256], [66, 95]],
    [[81, 89], [218, 247], [221, 246], [122, 199]],
    [[90, 152], [273, 297], [179, 210], [74, 156]],
    [[108, 111], [203, 222], [209, 267], [92, 131]],
    [[90, 99], [183, 244], [210, 251], [104, 148]],
    [[122, 90], [141, 183], [294, 304], [114, 124]],
    [[117, 115], [206, 213], [207, 244], [60, 90]],
    [[101, 97], [168, 238], [183, 238], [99, 148]],
    [[132, 143], [188, 223], [192, 233], [58, 88]],
], dtype=torch.float32)

print("setores shape:", setores.shape, "-> 20 setores x 4 faixas x 2 sexos")

total_classificado_por_setor = setores.sum(dim=(1, 2))
core_fitness_por_setor = setores[:, 1, :].sum(dim=1)
pct_core_fitness = core_fitness_por_setor / total_classificado_por_setor

top3 = torch.argsort(pct_core_fitness, descending=True)[:3]
for i in top3:
    print(f"{setores_id[i]:<17} {pct_core_fitness[i].item():>7.1%}")

# conferência: mesmo ranking do notebook 01 (230440005180085 com 57.0%)

## 5. O jeito idiomático: `nn.Module` + `torch.optim`

Ninguém escreve o loop de treino do Passo 8 do notebook 01 à mão em produção. O jeito normal em PyTorch é declarar o neurônio como um `nn.Module` (aqui, uma camada `nn.Linear(2, 1)` seguida de sigmoide) e deixar um `optimizer` cuidar da atualização dos pesos — a lógica interna é idêntica à do Passo 8, só empacotada.

Usamos o mesmo dado sintético do notebook 01 (mesmos pesos verdadeiros `[0.4, 0.6]`, viés `-0.3`, mesmas faixas de valor) — sem captação real observada ainda, o alerta do Passo 8 continua valendo aqui.

In [ ]:
class Neuronio(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.camada = torch.nn.Linear(2, 1)  # já contém w e b internamente

    def forward(self, x):
        return torch.sigmoid(self.camada(x)).squeeze(-1)


# DADO SINTÉTICO — mesma ressalva do Passo 8 do notebook 01: não é captação real observada
torch.manual_seed(42)
n_amostras = 200
pesos_verdadeiros = torch.tensor([0.4, 0.6])
vies_verdadeiro = -0.3

x1 = torch.empty(n_amostras).uniform_(0.10, 0.20)
x2 = torch.empty(n_amostras).uniform_(0.70, 1.40)
X_treino = torch.stack([x1, x2], dim=1)
ruido = torch.randn(n_amostras) * 0.05
y_alvo = torch.sigmoid(X_treino @ pesos_verdadeiros + vies_verdadeiro + ruido)

modelo = Neuronio()
perda_fn = torch.nn.MSELoss()
otimizador = torch.optim.SGD(modelo.parameters(), lr=0.5)

for epoca in range(2000):
    otimizador.zero_grad()
    pred = modelo(X_treino)
    perda = perda_fn(pred, y_alvo)
    perda.backward()
    otimizador.step()

    if epoca % 400 == 0:
        w_atual = modelo.camada.weight.detach().numpy().flatten()
        b_atual = modelo.camada.bias.item()
        print(f"época {epoca:>4}  perda={perda.item():.5f}  w={w_atual}  b={b_atual:.3f}")

w_final = modelo.camada.weight.detach().numpy().flatten()
b_final = modelo.camada.bias.item()
print(f"\npesos aprendidos: {w_final}   (verdadeiros: {pesos_verdadeiros.numpy()})")
print(f"viés aprendido  : {b_final:.3f}   (verdadeiro: {vies_verdadeiro})")

### Validação: bate com o Passo 5?

Mesma checagem do notebook 01 — os pesos numéricos podem sair diferentes dos verdadeiros (é o mesmo fenômeno de identificabilidade explicado lá), o que importa é a previsão final.

In [ ]:
with torch.no_grad():
    prioridade_pytorch = modelo(entradas)

for regiao, p_mao, p_treinado in zip(regioes, prioridade, prioridade_pytorch):
    print(f"{regiao:<13} prioridade(pesos à mão)={p_mao.item():.3f}   prioridade(PyTorch treinado)={p_treinado.item():.3f}")

## Onde isso te leva a seguir

1. **Mais camadas**: trocar `nn.Linear(2, 1)` por `nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())` — uma rede de verdade, com camada escondida. A lógica de treino (`optimizer.zero_grad()` → `loss.backward()` → `optimizer.step()`) não muda nada.
2. **Dado real de treino**: assim que existir captação de campanha medida por região/bairro, substituir `y_alvo` sintético por esse dado é a mudança que transforma isso de exercício em ferramenta — os Passos 7 e 9 já mostraram como puxar população real; falta o resultado observado do outro lado da equação.
3. **GPU**: `modelo.to("cuda")` e `X_treino.to("cuda")` bastam para rodar na placa de vídeo, se o volume de dados justificar (bairro a bairro, nacional, não faz diferença hoje — vale quando o dado escalar para milhões de exemplos).